In [1]:
# =========================================
# Cell 1 — Minimal install (Colab-safe)
# =========================================
!pip -q install --upgrade-strategy only-if-needed \
  openai datasets huggingface_hub tqdm matplotlib


In [2]:
# =========================================
# Cell 2 — Keys (paste in)
# =========================================
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY (hidden): ").strip()
os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN (required for gated FPB): ").strip()

assert os.environ["OPENAI_API_KEY"], "Missing OPENAI_API_KEY"
assert os.environ["HF_TOKEN"], "Missing HF_TOKEN"
print("✅ Keys set.")


Enter OPENAI_API_KEY (hidden): ··········
Enter HF_TOKEN (required for gated FPB): ··········
✅ Keys set.


In [3]:
# =========================================
# Cell 3 — Load FPB dataset (TheFinAI/en-fpb)
# =========================================
from datasets import load_dataset
import os

DATASET_NAME = "TheFinAI/en-fpb"

# Try common splits; if your dataset uses different split names, we'll print them.
try:
    ds_all = load_dataset(DATASET_NAME, token=os.environ["HF_TOKEN"])
    print("✅ Available splits:", list(ds_all.keys()))
except Exception as e:
    print("❌ Could not list splits. Error:\n", e)
    raise

SPLIT = "test" if "test" in ds_all else ("validation" if "validation" in ds_all else "train")
ds = ds_all[SPLIT]

print("✅ Using split:", SPLIT)
print(ds)
print("Columns:", ds.column_names)
print("Row0:", ds[0])


README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

✅ Available splits: ['train', 'test', 'valid']
✅ Using split: test
Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Row0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — ChatGPT-4 (chatgpt-4o-latest) predictor + helpers
# =========================================
import re, time
from openai import OpenAI

MODEL_ID = "chatgpt-4o-latest"  # ChatGPT-4 model used in ChatGPT
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def build_prompt(text: str, choices):
    return (
        "Classify the sentiment of the following financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = model_text.strip().lower()

    for c in choices:  # exact
        if s == str(c).strip().lower():
            return str(c)

    for c in choices:  # whole-word
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def chatgpt4_predict(prompt: str, max_retries: int = 6, sleep_base: float = 1.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.responses.create(model=MODEL_ID, input=prompt)
            text_out = getattr(resp, "output_text", None)

            usage = getattr(resp, "usage", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "input_tokens": getattr(usage, "input_tokens", None),
                    "output_tokens": getattr(usage, "output_tokens", None),
                    "total_tokens": getattr(usage, "total_tokens", None),
                }

            meta = {"id": getattr(resp, "id", None), "usage": usage_dict}
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}

def normalize_fpb_example(ex: dict):
    # text field (FPB variants differ)
    text_key = None
    for k in ["text", "sentence", "content", "input"]:
        if k in ex:
            text_key = k
            break
    if text_key is None:
        raise KeyError(f"No text field found. Keys={list(ex.keys())}")
    text = ex[text_key]

    # If dataset already provides choices+gold
    if "choices" in ex and ("gold" in ex or "label" in ex):
        choices = list(ex["choices"])
        gold = int(ex.get("gold", ex.get("label")))
        return text, choices, gold

    # Default FPB mapping
    choices = ["negative", "neutral", "positive"]
    lab = ex.get("label", ex.get("gold", ex.get("sentiment", None)))
    if lab is None:
        raise KeyError(f"No label/gold/sentiment found. Keys={list(ex.keys())}")

    if isinstance(lab, str):
        lab_norm = lab.strip().lower()
        if lab_norm not in choices:
            raise ValueError(f"Unknown string label: {lab}")
        gold = choices.index(lab_norm)
    else:
        gold = int(lab)
        if gold < 0 or gold >= len(choices):
            raise ValueError(f"Label int {gold} out of range for {choices}")

    return text, choices, gold


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None         # None = FULL split; set 200 for quick test
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.25 # raise to 0.5–1.0 if rate-limited

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"chatgpt4_{MODEL_ID}_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text, choices, gold = normalize_fpb_example(ex)

        prompt = build_prompt(text, choices)
        model_out, meta = chatgpt4_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,  # JSON-safe
        }

        # safe JSON write
        try:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        except TypeError:
            rec["meta"] = {"meta_str": str(meta)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 970/970 [28:22<00:00,  1.76s/it]

✅ Saved raw responses to: text_responses/chatgpt4_chatgpt-4o-latest_TheFinAI_en-fpb_test.jsonl


In [8]:
# =========================================
# Cell 6 — Metrics (sklearn-like classification report) + confusion matrix (NO sklearn)
# =========================================
import json, os, math

# Use the same variable you used earlier:
# raw_path = "text_responses/....jsonl"

all_rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            try:
                all_rows.append(json.loads(line))
            except:
                pass

assert len(all_rows) > 0, f"No rows found in {raw_path}"

# Infer label names from saved file (assumes consistent choices)
choices0 = all_rows[0].get("choices", ["negative", "neutral", "positive"])
for r in all_rows[:50]:
    if r.get("choices", choices0) != choices0:
        raise ValueError("Inconsistent 'choices' across rows. Tell me and I’ll adapt the code.")

idx_to_label = {i: lab for i, lab in enumerate(choices0)}
labels = list(range(len(choices0)))

gold = [int(r["gold"]) for r in all_rows]
pred = [int(r.get("predicted_index", -1)) for r in all_rows]  # -1 = invalid/unparsed

# Valid-only pairs for CM and per-class metrics (like typical reporting)
pairs = [(g, p) for g, p in zip(gold, pred) if p >= 0]
gold_v = [g for g, _ in pairs]
pred_v = [p for _, p in pairs]

def safe_div(a, b):
    return a / b if b else 0.0

# Per-class counts
support = {c: 0 for c in labels}
tp = {c: 0 for c in labels}
fp = {c: 0 for c in labels}
fn = {c: 0 for c in labels}

for yt, yp in pairs:
    support[yt] += 1
    for c in labels:
        if yp == c and yt == c:
            tp[c] += 1
        elif yp == c and yt != c:
            fp[c] += 1
        elif yp != c and yt == c:
            fn[c] += 1

prec = {}
rec = {}
f1 = {}
for c in labels:
    prec[c] = safe_div(tp[c], tp[c] + fp[c])
    rec[c]  = safe_div(tp[c], tp[c] + fn[c])
    f1[c]   = safe_div(2 * prec[c] * rec[c], prec[c] + rec[c])

# Accuracy: report BOTH "all" (invalid counted wrong) and "valid-only"
acc_all = safe_div(sum(int(a == b) for a, b in zip(gold, pred)), len(gold))
acc_valid = safe_div(sum(int(a == b) for a, b in pairs), len(pairs)) if pairs else None

total_support = sum(support.values())

macro_p = sum(prec[c] for c in labels) / len(labels)
macro_r = sum(rec[c]  for c in labels) / len(labels)
macro_f = sum(f1[c]   for c in labels) / len(labels)

w_p = safe_div(sum(prec[c] * support[c] for c in labels), total_support)
w_r = safe_div(sum(rec[c]  * support[c] for c in labels), total_support)
w_f = safe_div(sum(f1[c]   * support[c] for c in labels), total_support)

# Confusion matrix (valid-only)
cm = [[0 for _ in labels] for __ in labels]
for g, p in pairs:
    cm[g][p] += 1

# Print like your screenshot
print(f"Accuracy:  {acc_valid:.4f}" if acc_valid is not None else f"Accuracy:  N/A (no valid preds)")
print("\nClassification report:\n")
name_width = max(len(idx_to_label[c]) for c in labels + [labels[0]])
print(f"{'':>{name_width}}  {'precision':>9}  {'recall':>7}  {'f1-score':>8}  {'support':>7}\n")

for c in labels:
    name = idx_to_label[c]
    print(f"{name:>{name_width}}  {prec[c]:9.4f}  {rec[c]:7.4f}  {f1[c]:8.4f}  {support[c]:7d}")

print()
if acc_valid is not None:
    print(f"{'accuracy':>{name_width}}  {'':>9}  {'':>7}  {acc_valid:8.4f}  {total_support:7d}")
else:
    print(f"{'accuracy':>{name_width}}  {'':>9}  {'':>7}  {'N/A':>8}  {total_support:7d}")

print(f"{'macro avg':>{name_width}}  {macro_p:9.4f}  {macro_r:7.4f}  {macro_f:8.4f}  {total_support:7d}")
print(f"{'weighted avg':>{name_width}}  {w_p:9.4f}  {w_r:7.4f}  {w_f:8.4f}  {total_support:7d}")

# Save metrics JSON (includes both accuracies + report numbers + CM)
metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": len(all_rows),
    "num_valid_predictions": len(pairs),
    "accuracy_all": acc_all,
    "accuracy_valid_only": acc_valid,
    "per_class": {
        idx_to_label[c]: {
            "precision": prec[c],
            "recall": rec[c],
            "f1": f1[c],
            "support": support[c],
        } for c in labels
    },
    "macro_avg": {"precision": macro_p, "recall": macro_r, "f1": macro_f, "support": total_support},
    "weighted_avg": {"precision": w_p, "recall": w_r, "f1": w_f, "support": total_support},
    "confusion_matrix_labels": [idx_to_label[c] for c in labels],
    "confusion_matrix": cm,
}

os.makedirs("evaluation", exist_ok=True)
metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\n✅ Metrics saved:", metrics_path)


Accuracy:  0.8093

Classification report:

          precision   recall  f1-score  support

positive     0.6705   0.8448    0.7476      277
 neutral     0.9125   0.7591    0.8288      577
negative     0.8014   0.9741    0.8794      116

accuracy                        0.8093      970
macro avg     0.7948   0.8593    0.8186      970
weighted avg     0.8301   0.8093    0.8116      970

✅ Metrics saved: evaluation/chatgpt4_chatgpt-4o-latest_TheFinAI_en-fpb_test_metrics.json
